In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sktime.forecasting.auto_reg import AutoREG
from sktime.utils.plotting import plot_series
from sktime.performance_metrics.forecasting import MeanAbsoluteError

In [14]:
data = pd.read_csv("../data/data.csv")
data["Period"] = pd.to_datetime(data["Period"].astype(str), format="%Y%m")
data["Period"] = pd.PeriodIndex(data["Period"], freq="M")
data = data.set_index("Period").sort_index()

y = data["Immigration"]
holdout_size = 6

y_development = y.iloc[:-holdout_size]
y_holdout = y.iloc[-holdout_size:]

print(f"Development: {y_development.index.min()} to {y_development.index.max()}")
print(f"Holdout: {y_holdout.index.min()} to {y_holdout.index.max()}")

Development: 2010-01 to 2025-06
Holdout: 2025-07 to 2025-12


In [ ]:

def rolling_origin_backtest(y, lags, horizon, min_train_size=36, step=1):

    mae = MeanAbsoluteError()
    scores = []

    for train_end in range(min_train_size, len(y) - horizon + 1, step):

        # Expanding training window
        y_train = y.iloc[:train_end]
        y_test = y.iloc[train_end:train_end + horizon]

        # Fit autoregressive model
        model = AutoREG(lags=lags, trend="c")
        model.fit(y_train)

        # Forecast the next h months
        fh = np.arange(1, horizon + 1)
        y_pred = model.predict(fh=fh)

        # Evaluate forecast
        scores.append(mae(y_test, y_pred))

    if not scores:
        raise ValueError("Not enough data for the requested backtest.")

    return np.mean(scores)


In [ ]:

lags_to_test = [12, 18]
horizons = [1, 3, 6]

results = []

for lags in lags_to_test:
    for horizon in horizons:
        score = rolling_origin_backtest(
            y=y_development,
            lags=lags,
            horizon=horizon,
            min_train_size=60,
            step=1
        )

        results.append({
            "lags": lags,
            "horizon": horizon,
            "mae": score
        })

results = pd.DataFrame(results)

# Compare performance across horizons
lag_scores = (
    results.groupby("lags")["mae"]
    .mean()
    .sort_values()
)

# Select one overall lag order
best_lags = int(lag_scores.idxmin())

print(results)
print("\nAverage MAE by lag order:")
print(lag_scores)
print(f"\nSelected lag order: {best_lags}")


   lags  horizon         mae
0    12        1  203.518931
1    12        3  214.422205
2    12        6  217.637510
3    18        1  189.570529
4    18        3  200.473554
5    18        6  211.980132

Average MAE by lag order:
lags
18    200.674738
12    211.859549
Name: mae, dtype: float64

Selected lag order: 18


In [19]:

# Fit the selected baseline
final_model = AutoREG(
    lags=best_lags,
    trend="c"
)

final_model.fit(y_development)

# Evaluate each forecast horizon
holdout_results = []
predictions = {}

mae = MeanAbsoluteError()

for horizon in horizons_to_test:

    fh = np.arange(1, horizon + 1)

    y_pred = final_model.predict(fh=fh)
    y_test = y_holdout.iloc[:horizon]

    score = mae(y_test, y_pred)

    predictions[horizon] = y_pred

    holdout_results.append({
        "horizon": horizon,
        "lags": best_lags,
        "mae": score
    })

holdout_results = pd.DataFrame(holdout_results)

print(holdout_results)


   horizon  lags         mae
0        1    18  188.659069
1        3    18  130.396108
2        6    18  203.473564
